# TTA Example

> Usage

```bash
uv run jupyter nbconvert --to script example.ipynb
uv run example.py --dataset shift --model rcnn --method norm_engine --device 0
```

## Imports and Configs

In [ ]:
import sys
from argparse import ArgumentParser
from contextlib import redirect_stdout, nullcontext
from os import path, environ, system, makedirs, devnull

import torch

from ttadapters import datasets
from ttadapters import models, methods
from ttadapters.datasets import scenarios
from ttadapters.utils.validator import DetectionEvaluator
from ttadapters.utils.visualizer import visualize_metrics

In [ ]:
import pandas as pd

pd.options.display.float_format = lambda x: f"{x*100 if x < 1 else x:.4f}"

### Parse Arguments

In [ ]:
ABLATION_NUM = 2

In [ ]:
# Set Batch Size
BATCH_SIZE = 1  # Online
FIT_BATCH_SIZE = 40

# Set CUDA Device Number
DEVICE_NUM = 0

# Set Total Rounds
TOTAL_ROUNDS = 1

# Set Data Root
DATA_ROOT = path.join(".", "data")
RESULT_ROOT = path.join(".", "results", "ablations", f"aab{ABLATION_NUM}")

# Set Target Dataset
SOURCE_ZOO = ["shift", "city"]
SOURCE_DOMAIN = datasets.SHIFTDataset
SOURCE_DATASET = SOURCE_ZOO[0]

# Set Target Scenario
SCENARIO_ZOO = ["continual_tta", "gradual_tta"]
SCENARIO_TYPE = SCENARIO_ZOO[0]

# Set Model List
MODEL_ZOO = ["rcnn", "swinrcnn", "yolo11", "rtdetr"]
MODEL_TYPE = MODEL_ZOO[0]

# Set method
METHOD_ZOO = ["gita_engine"]
METHOD_TYPE = METHOD_ZOO[0]

In [ ]:
# Create argument parser
parser = ArgumentParser(description="Adaptation experiment script for Test-Time Adapters")

# Add model arguments
parser.add_argument("--dataset", type=str, choices=SOURCE_ZOO, default=SOURCE_DATASET, help="Target dataset")
parser.add_argument("--scenario", type=str, choices=SCENARIO_ZOO, default=SCENARIO_TYPE, help="Target TTA scenario")
parser.add_argument("--model", type=str, choices=MODEL_ZOO, default=MODEL_TYPE, help="Model architecture")
parser.add_argument("--method", type=str, choices=METHOD_ZOO, default=METHOD_TYPE, help="Method")

# Add training arguments
parser.add_argument("--adapt-batch", type=int, default=BATCH_SIZE, help="Adaptation batch size")
parser.add_argument("--fit-batch", type=int, default=FIT_BATCH_SIZE, help="Engine fitting batch size")
parser.add_argument("--total-rounds", type=int, default=TOTAL_ROUNDS, help="Total rounds")
parser.add_argument("--data-root", type=str, default=DATA_ROOT, help="Root directory for datasets")
parser.add_argument("--results-root", type=str, default=RESULT_ROOT, help="Root directory for adaptation results")
parser.add_argument("--device", type=int, default=0, help="CUDA device number")
parser.add_argument("--disable-datalog", action="store_true", help="Disable datalog")

# Parsing arguments
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
    print("INFO: Running in notebook mode with default arguments")
else:
    args = parser.parse_args()

# Configure device
DEVICE_NUM = 0 if not args.device else args.device
environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE_NUM)
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1
print(f"INFO: Using device - {device}:{DEVICE_NUM}")

# Update global variables based on parsed arguments
BATCH_SIZE = args.adapt_batch
FIT_BATCH_SIZE = args.fit_batch
TOTAL_ROUNDS = args.total_rounds
DATA_ROOT = args.data_root
DISABLE_DATALOG_CTX = redirect_stdout(open(devnull, "w"))
RESULT_ROOT = args.results_root
MODEL_TYPE = args.model
METHOD_TYPE = args.method
SCENARIO_TYPE = args.scenario
SOURCE_DATASET = args.dataset
match args.dataset:
    case "shift":
        SOURCE_DOMAIN = datasets.SHIFTDataset
        import logging
        logging.getLogger("shift_dev_logger").setLevel(logging.CRITICAL)
    case "city":
        SOURCE_DOMAIN = datasets.CityScapesDataset
    case _:
        raise ValueError(f"Unsupported dataset: {args.dataset}")

print(f"INFO: Running online adaptation with batch size {BATCH_SIZE}")

In [ ]:
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

## Define Dataset

In [ ]:
# Fast download patch
datasets.patch_fast_download_for_object_detection()

In [ ]:
# Ensure split (required due to Scenario class works with coroutines)
with DISABLE_DATALOG_CTX:
    match SOURCE_DOMAIN:
        case datasets.SHIFTDataset:
            train_dataset = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)
        case datasets.CityScapesDataset:
            train_dataset = datasets.CityScapesDatasetForObjectDetection(root=DATA_ROOT, train=True)
        case _:
            raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

# Dataset info
CLASSES = train_dataset.classes
NUM_CLASSES = len(CLASSES)
print(f"INFO: Number of classes - {NUM_CLASSES} {CLASSES}")

## Load Base Model

In [ ]:
# Initialize base_model
DATA_TYPE = torch.float32
match MODEL_TYPE:
    case "rcnn":
        base_model = models.FasterRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "swinrcnn":
        base_model = models.SwinRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "yolo11":
        base_model = models.YOLO11ForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "rtdetr":
        DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.RTDetrForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case _:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")

data_preparation = base_model.DataPreparation(train_dataset, evaluation_mode=True)
print(f"INFO: Using data precision - {DATA_TYPE}")
print("INFO: Model state loaded -", load_result)
base_model.to(device)

## Load Adaptation Method

In [ ]:
from ttadapters.methods.cascaded import GITAEngine, GITAConfig

In [ ]:
from typing import Literal
from dataclasses import dataclass, field
from enum import Enum


class TargetKeyPreset(Enum):
    """
    Preset patterns for cascade_target.
    Strategies are applied to the **EARLY Blocks** of the backbone to fast optimization.
    """
    RESNET_LATE = [  # ResNet: res5 (3 blocks) / stages 0 = 3 blocks
        r"\.res5.*\.conv[123]\.norm$",  # BottleneckBlock (Detectron2)
        r"\.stages\.3.*\.layer\.[012]\.normalization$"  # RTDetrResNetBottleNeckLayer (RT-DETR)
    ]
    SWIN_LATE = [  # Swin: layer0 (2 blocks) + layer1 (2 blocks) = 4 blocks
        r"\.layers\.2\.blocks\.[45]\.norm[12]$",  # SwinTransformerBlock
        r"\.layers\.3\.blocks\..*\.norm[12]$"  # SwinTransformerBlock
    ]
    YOLO11_STAGE1 = [  # YOLO11: Conv stride2 + C3k2 #1 (1/4 scale: model.1~2)
        r"(^|\.)model\.[12]\..*bn$"
    ]
    YOLO11_STAGE2 = [  # YOLO11: Conv stride2 + C3k2 #2 (1/8 scale: model.3~4)
        r"(^|\.)model\.[34]\..*bn$"
    ]
    YOLO11_STAGE3 = [  # YOLO11: Conv stride2 + C3k2 #3 (1/16 scale: model.5~6)
        r"(^|\.)model\.[56]\..*bn$"
    ]
    YOLO11_STAGE4 = [  # YOLO11: Conv stride2 + C3k2 #4 (1/32 scale: model.7~8)
        r"(^|\.)model\.[78]\..*bn$"
    ]


@dataclass
class GITAConfig(GITAConfig):
    @classmethod
    def from_preset(cls, base_model, **kwargs):
        """Create configuration from preset."""
        from ttadapters.models import (
            FasterRCNNForObjectDetection, SwinRCNNForObjectDetection,
            RTDetrForObjectDetection, YOLO11ForObjectDetection
        )
        if isinstance(base_model, FasterRCNNForObjectDetection):
            return cls(cascade_target=TargetKeyPreset.RESNET_LATE.value, **kwargs)
        elif isinstance(base_model, SwinRCNNForObjectDetection):
            return cls(cascade_target=TargetKeyPreset.SWIN_LATE.value, **kwargs)
        elif isinstance(base_model, RTDetrForObjectDetection):
            return cls(cascade_target=TargetKeyPreset.RESNET_LATE.value, **kwargs)
        elif isinstance(base_model, YOLO11ForObjectDetection):
            return cls(
                cascade_target=TargetKeyPreset.YOLO11_STAGE1.value,
                masked_processing=True, mask_value=114, **kwargs
            )
        else:
            raise ValueError(f"Unsupported base model type: {type(base_model)}")

In [ ]:
config = GITAConfig.from_preset(base_model=base_model)

In [ ]:
adaptive_model = GITAEngine(config, base_model=base_model)
adaptive_model.to(device)

### Fit engine with source if required

In [ ]:
adaptive_model.fit(data_preparation, batch_size=FIT_BATCH_SIZE, shuffle=False)

## Evaluation

In [ ]:
# Load Pretrained APT Weights & Un-Freeze Model Encoder
# Allow FPN/Encoder to adapt during online adaptation
base_model.eval()
adaptive_model.online()

In [ ]:
def get_save_path(scenario, round=None):
    suffix = f"_r{round}" if round else ""
    batch = f"_b{BATCH_SIZE}"
    prefix = path.join(RESULT_ROOT, MODEL_TYPE, scenario.__class__.__name__.lower(), adaptive_model.model_type)
    makedirs(prefix, exist_ok=True)
    return path.join(prefix, "model" + batch + suffix + ".pkl"), path.join(prefix, "result" + batch + suffix + ".json")

In [ ]:
import json

def make_json_serializable(result):
    if isinstance(result, list):
        return [make_json_serializable(item) for item in result]
    elif isinstance(result, dict):
        return {(k.value if hasattr(k, 'value') else k): make_json_serializable(v) for k, v in result.items()}
    return result

def save_result_json(result, result_path):
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

### Load Scenarios

In [ ]:
continual_scenario, gradual_scenario, standard_scenario, universal_scenario = None, None, None, None

with DISABLE_DATALOG_CTX:
    match SOURCE_DOMAIN:
        case datasets.SHIFTDataset:
            if SCENARIO_TYPE == "continual_tta":
                continual_scenario = scenarios.SHIFTDiscreteScenarioForContinualTTA(
                    root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
                )
            elif SCENARIO_TYPE == "gradual_tta":
                gradual_scenario = scenarios.SHIFTContinuousScenarioForGradualTTA(
                    root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
                )
        case datasets.CityScapesDataset:
            if SCENARIO_TYPE == "continual_tta":
                continual_scenario = scenarios.CityScapesDiscreteScenarioForContinualTTA(
                    root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
                )
            elif SCENARIO_TYPE == "gradual_tta":
                gradual_scenario = scenarios.CityScapesContinuousScenarioForGradualTTA(
                    root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
                )
        case _:
            raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

### Setup TTA

In [ ]:
tta = methods.MethodContainer(**{
    adaptive_model.model_name: adaptive_model
})

### Continual TTA - Go rounds without reset

In [ ]:
evaluator = DetectionEvaluator(tta.methods(), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False)
evaluator_loader_params = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
continual_result = []
if continual_scenario:
    for rd, this in tta.go_rounds(end_round=TOTAL_ROUNDS):
        result = visualize_metrics(continual_scenario(**evaluator_loader_params).play(evaluator, index=this))
        continual_result.append(result)
        _, result_path = get_save_path(continual_scenario, rd)
        save_result_json(result, result_path)

### Gradual TTA - With resets

In [ ]:
evaluator = DetectionEvaluator(tta.methods(), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False, required_reset=True)
evaluator_loader_params = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
if gradual_scenario:
    gradual_result = visualize_metrics(gradual_scenario(**evaluator_loader_params).play(evaluator, index=tta.names(round=1)))
    _, result_path = get_save_path(gradual_scenario)
    save_result_json(gradual_result, result_path)